In [2]:
import json
import re
import ntpath
import os
import pandas as pd

In [91]:
parties = ['CDU/CSU', 'GRÜNE','SPD', 'FDP', 'AfD', 'DIE LINKE', 'KPD', 'BP', 'DP', 'WAV', 'Z', 'fraktionslos', ] # Die Abkürzungen der wichtigsten - auch historischen - Parteien
partei_match = r'(CDU\/CSU|CSU|CDU|GRÜNE|FDP|AfD|SPD|KPD|BP|DP|WAV|Z|DIE LINKE|fraktionslos)'

In [92]:
def load_text(path_to_json):
    with open(path_to_json) as f:
        dictionary = json.load(f)
        if 'text' in dictionary:
            text = dictionary['text']
        else:
            return 'no text found'
    text = re.sub(r'\n-\n', '-', text) # weird formatting probably due to digitalization from printed mediums
    text = re.sub(r'DIE\s*?LINKE|PDS/\s*?Linke\s*?Liste|PDS/LL', 'DIE LINKE', text) # Fusion von PDS und linke Liste
    text = re.sub(r'F\.D\.P\.', 'FDP', text) # Was auch immer... kommt aber manchmal vor
    buendnis_match = re.compile(r'(BÜND(-\n?)?NIS((-\n?)?SES)?\s*?90\/\s*(DIE\s*?)?GRÜ(-\n?)?NEN?)', flags=re.UNICODE|re.IGNORECASE) # Ja, das braucht man...
    text = re.sub(buendnis_match,'GRÜNE', text) # Deklinationen etc.etc.etc.
    return text

# Regular Expressions

### Namen
Um Namen zu erkennen, mussten wir alle Eventualitäten und Konventionen der Protokollanten abbilden. Außerdem gibt es ein paar echt interessante Einzelfälle...

In [ ]:
name_match = ( '('
              +  r'(?:Dr\.\s)?'   # Optional "Dr. "
              + r'(?:\w+(?:-\w+)?\s)'   # First name (or hyphenated first name)
              + r'(?:\w+\.\s)?'   # Optional middle initial
              + r'(?:von\s)?'      # Optional Title
              + r'\w+(?:-\w+)?'   # Last name (or hyphenated last name)
              + ')'
             )

In [89]:
print(re.findall(name_match, 'Dr. Harald J. Töpfer'))
print(re.findall(name_match, 'Dr. Marie-Agnes Strack-Zimmermann'))
print(re.findall(name_match, 'Brynden B. Tully'))
print(re.findall(name_match, 'Oberyn Martell'))

['Dr. Harald J. Töpfer']
['Dr.\xa0Marie-Agnes Strack-Zimmermann']
['Brynden B. Tully']
['Oberyn Martell']


### Unterbrechungen

Unterbrechungen sind Einwürfe aus den Reihen des Bundestags, bei denen der Sprecher erkennbar war und deswegen mit vermerkt wurde. 

In [101]:
unterbrechung_match = re.compile(rf'{name_match}\s+' # interruptions are comprised of a name
                                rf'\[{partei_match}\]:\s' # ...followed by the party in square brackets
                                r'([\s\S]*?)' # ...followed by the comment text including newlines
                                r'[-–—\)]', # ...ended by some sort of hyphen or closing round brackets
                                flags = re.UNICODE)

In [96]:
kommentar_beispiel = """...mit Ihren Ankündigungen zu höheren Lebensmittelpreisen erreichen Sie nur eines: Sie sorgen für Verunsicherung.
(Beifall bei der CDU/CSU – Renate Künast [GRÜNE]: Wie? Das hat doch Klöckner auch getan! – Harald Ebner [GRÜNE]: Das ist doch peinlich!)"""
re.findall(unterbrechung_match, kommentar_beispiel)

[('Renate Künast', 'GRÜNE', 'Wie? Das hat doch Klöckner auch getan!\xa0'),
 ('Harald Ebner', 'GRÜNE', 'Das ist doch peinlich!')]

### Zurufe

Zurufe sind Unterbrechungen des Redners, bei denen der genaue Unterbrecher nicht mitprotokolliert wurde. Diese zu finden ist auch nicht sonderlich schwierig.

In [ ]:
zuruf_match = re.compile(r'[-–—\s\(]'   # Interruptions with unknown speakers but known party affiliation are comprised of a leading hyphen or opening parenthesis
                         r'Zuruf .*?'   # ... followed by 'Zuruf' and some more irrelevant fillwords
                         rf'{partei_match}:\s'  # ... folllowed by the party name and :
                         r'([\s\S]*?)'     # ... followed by the text of the comment
                         r'[-–—\)]',    # ... ended by some sort of hyphen or closing round parenthesis
                        flags = re.UNICODE)

In [100]:
zuruf_beispiel = """Danach, Frau Kollegin Matthäus-Maier, wurde es 
leider wieder langweilig, enttäuschend: das gleiche 
Ritual, 
(Zuruf von der CDU/CSU: Immer das
selbe!) 
leider auch immer wieder Unterstellungen. Bei Ihnen 
taucht sehr oft das Wort „ehrlich" auf. 
(Zuruf von der CDU/CSU: Das ist immer ver
dächtig!) 
Sie sollten es allerdings nicht permanent mit
Unterstellungen"""

re.findall(zuruf_match, zuruf_beispiel)

[('CDU/CSU', 'Immer das\nselbe!'), ('CDU/CSU', 'Das ist immer ver\ndächtig!')]

### Beifall

Interessanterweise wird in den Protokollen auch Beifall vermerkt. Diese Daten lassen wir natürlich nicht unangetastet.

In [97]:
beifall_match = re.compile(r'[-–—\s\(]' # Applause is comprised of a leading hyphen or opening parenthesis
                           'Beifall'    # ...followed by "Beifall"
                           r'[\s\S]*?'  # ...followed by a sentence including the party names
                           r'[-–—\)]',  # ...ended by some sort of hyphen or closing round parenthesis
                           flags = re.UNICODE)

In [99]:
beifall_beispiel = """(Beifall bei der CDU/CSU sowie bei
Abgeordneten der SPD, des GRÜNE und der FDP – Stephan Brandner 
[AfD]: Erbärmliche Wendehälse!)"""
re.findall(beifall_match, beifall_beispiel)


['(Beifall bei der CDU/CSU sowie bei\nAbgeordneten der SPD, des GRÜNE und der FDP –']

Der gematchte String wird dann später weiterverarbeitet. Um die Beifall gebenden Parteien zu finden, sieht man einfach nach, ob deren Kürzel im String vorkommt.  
Man sieht hier auch die Ersetzung von "Bündnisses 90/die Grünen" in der load_text() Methode, was wie hier zu fehlerhafter Grammatik führen kann.

### Redebeiträge/Sprecher

Damit zu jeder Unterbrechung auch vermerkt werden kann, wer denn überhaupt unterbrochen wurde, muss das natürlich auch erkannt werden. Das war aber leichter gesagt, als getan.

speaker_match ist ein RegEx, der immer nur Stück für Stück erweitert werden konnte, als wieder eine neue Eventualität nicht vom vorigen RegEx erfasst wurde.
Im Git finden sich bestimmt 10 unterschiedliche Versionen. Es kann auch sein, dass mit dieser Variante ein paar Sonderfälle nicht zu 100% abgedeckt sind.  
Für kleinere RegEx konnte der Chatbot unseres Vertrauens noch sinnvolle Ausgaben machen (z.B. beim Namen gingen Teile davon noch ganz gut).
Die Aufgabe hier war aber deutlich zu

Es gibt mehrere Varianten, wie eine Rede gestartet werden kann. 
* Variante 1 - die leicht Erkennbare:  
  * Ein Name (name_match) gefolgt von runden Klammern, in denen das Parteikürzel steht  
  * **Beispiele:** 
    * >Ingo Gädechens (CDU/CSU):  
      >Frau Präsidentin! Liebe Kolleginnen und Kollegen! ...
    * >René Springer (AfD):  
      >Vielen Dank, Frau Präsidentin...

* Variante 2 - die schwierig Erkennbare:  
  * Staatssekretäre, Bundesminister, Bundeskanzler etc. werden mit ihrem Titel und ohne Angabe der Partei vermerkt.  
  Das bedeutet, dass...  
    1. man die Parteizugehörigkeit (soweit vorhanden) nachschlagen muss.
    2. die Erkennung deutlich schwieriger wird, weil die runden Klammern mit Parteizugehörigkeit nicht mehr da sind, um den RegEx einzuschränken.  
  Was bleibt ist Trial and Error, um letztendlich herauszufinden, dass zum Glück (hoffentlich) alle dieser Redeanfänge nur einen der folgenden Anfänge nach dem Komma haben  
    (?:Parl\. Staatssekretär|Staatssekretär|Bundeskanzler|Bundesminister)  
  Außerdem findet man erst heraus, dass .*? sich nicht zum Matchen der unbekannten Anteile, wie "beim Bundesminster des Inneren" eignet, wenn man den RegEx auch auf Protokolle aus dem 90ern anwendet.  
  Nach fast jeder Änderung muss man das Ganze auf 4 unterschiedlichen Protokollen im Zeitraum 2024 bis 1991 ausprobieren, um zu sehen, ob es denn noch passt.  
  Die jetzige Version lässt vom Namen bis zum ':' maximal eine Newline zu und ist deutlich simpler, als so manche Zwischenschritte auf dem Weg dahin.
  * **Beispiele** (mit akkurat gesetzten Newlines):  
    * >Manfred Kanther, Bundesminister des Innern: Frau Präsidentin!... 
    * >Fritz Rudolf Körper, Parl. Staatssekretär beim  
      >Bundesminister des Innern:  
      >Herr Kollege Grindel...
    * >Christian Lindner, Bundesminister der Finanzen:  
      >Sehr geehrte, liebe Frau Kollegin ...
  * **Antibeispiel** (mit akkurat gesetzten Newlines):  
    * >\n  
      >Arbeitsvertrag unterschrieben, und urplötzlich fällt Herrn Staatssekretär Graichen ein: Oh, das ist ja mein Trauzeuge.  
      
      Das hier hat z.B. in einer vorigen Variante gematcht. "Arbeitsvertrag unterschrieben" war der Name, "Staatssekretär" ist in den nächsten 100 Zeichen vorgekommen und ein ":" gab es auch noch.  




In [115]:
speaker_match = re.compile(
    rf'{name_match}\s(?:\(\w+\)\s)?\({partei_match}\):'  # Matches speaker's name, optional city name, followed by party name in parentheses, e.g., "Harald Töpfer (Party):"
    '|'
    r'\n'  # Newline
    rf'{name_match}, '
    r'((?:Parl\. Staatssekretär|Staatssekretär|Bundeskanzler|Bundesminister))[\w ]{0,100}\n?[\w ]{0,100}?:',  # Matches name followed by official title and fillwords with no more than one newline, e.g. "Harald Töpfer, Bundeskanzler" "Ronald Wiesel, Staatssekretär"
    flags=re.UNICODE
)
# bottom part of the regex (after |) has two capture-groups too so that the alignment of names is not out of order when splitting
# conveniently, no distinction between male and female Staatsminister(in) , Saatssekretär(in), Bundeskanzler(in) etc. has to be made as they start with the same chars
speaker_match

re.compile(r'((?:Dr\.\s)?(?:\w+(?:-\w+)?\s)(?:\w+\.\s)?(?:von\s)?\w+(?:-\w+)?)\s(?:\(\w+\)\s)?\((CDU\/CSU|CSU|CDU|GRÜNE|FDP|AfD|SPD|KPD|BP|DP|WAV|Z|DIE LINKE|fraktionslos)\):|\n((?:Dr\.\s)?(?:\w+(?:-\w+)?\s)(?:\w+\.\s)?(?:von\s)?\w+(?:-\w+)?), ((?:Parl\. Staatssekretär|Staatssekretär|Bundeskanzler|Bundesminister))[\w ]{0,100}\n?[\w ]{0,100}?:',
           re.UNICODE)

Ziemlich lang und unübersichtlich...

In [129]:
speaker_strack_zimmermann = 'Dr. Marie-Agnes Strack-Zimmermann (FDP): '
print(re.findall(speaker_match, speaker_strack_zimmermann ))

speaker_farle = '''Robert Farle
(fraktionslos):
Sehr geehrter Herr Präsident! Sehr'''
print(re.findall(speaker_match, speaker_farle))

speaker_koerper = '''
Rudolf Körper, Parl. Staatssekretär beim  
Bundesminister des Innern:  
Herr Kollege Grindel ...'''
print(re.findall(speaker_match, speaker_koerper))

speaker_scholz = '''
Olaf Scholz, Bundeskanzler:
Es ist sehr gut'''
print(re.findall(speaker_match, speaker_scholz ))

[('Dr. Marie-Agnes Strack-Zimmermann', 'FDP', '', '')]
[('Robert Farle', 'fraktionslos', '', '')]
[('', '', 'Rudolf Körper', 'Parl. Staatssekretär')]
[('', '', 'Olaf Scholz', 'Bundeskanzler')]


#### **Formate** 
Schwierig wurde die Entwicklung der RegEx vor allem dadurch, dass sich das Format in den Jahren bis 1991 mehrfach geändert hat. Die heutigen Protokolle sind meistens schön formatiert, haben mehrere Newlines vor einem Redeanfang und sind für die Sicht am Computer gemacht.
Die alten Protokolle wurden aber wie [hier](https://dserver.bundestag.de/btp/12/12093.pdf) in zwei Spalten gedruckt. Das führt zu sehr vielen Zeilenumbrüchen mitten in Sätzen. 

Der Grund, warum unsere Analysen nur bis 1991 zurückreichen ist, dass frühere Protokolle in einem gravierend anderen Format geschrieben sind. Hier wird immer nur der Nachname der Abgeordneten genannt. Um zusätzlich Namenskonflikte zu beheben wird dahinter in runden Klammern auch die Stadt/der Wahlkreis/der Geburtsort? angegeben. Wir haben ohnehin schon sehr viele Daten. Deswegen ist der Mehrwert für unser Projekt durch zusätzliche Daten von früher als 1991 relativ gering, weshalb wir uns dagegen entschieden haben noch einmal neue RegEx zu erstellen.  
Anmerkung: Das Format hat sich von 1949 bis 1991 noch mehrere Male geändert. Ein einzelner Extra-Satz RegEx hätte nicht gereicht.

### Fehler in den Protokollen 

Die Entwicklung der RegEx wurde auch noch durch teils gravierende Fehler in den Protokollen schwer gemacht.  
Manchmal sind es nur kleine Dinge. z.B. wenn der Protokollant/die Protokollantin nur CDU anstatt CDU/CSU schreibt. Das lässt sich ja relativ leicht lösen.  

Manchmal gehen aber ganze Teile des Protokolls kaputt.

**Beispiel:** Die JSON-Datei, die z.B. für das Protokoll am 13. April 2000 von der API bereitgestellt wurde. Der innerhalb der JSON-Datei gelieferte Text ist ohne Veränderung in ./data/examples als .txt Datei zu finden.
Als zum Informatikstudium passenden Abschnitt empfehle ich Zeile 3028-3050.  
Hier wurden die rechte und linke Seite des Quell-Drucks beim automatischen Einlesen zusammengefügt.  
Resultat ist ein Text, der mittendrin zwischen den beiden Spalten hin und herspringt.
<details>
  <summary>Text</summary>

>Diesen Lernprozess begrüßen wir sehr, wir begrüßen der Bundesministerin für Bildung und Forschung: Bitte.  
>auch den Sinneswandel von Frau Merkel, die am Montag  
>dieser Woche erklärt hat, die CDU habe keine grundsätz- Dr. Christa Luft (PDS): Herr Kollege Catenhusen, Sie  
>lichen Einwände gegen diese Aktion. Sicherlich ist Ihr haben eben betont, dass es in der IT-Branche vor allem  
>Menschen mit Hochschulabschluss geben müsse. Was sa- Menschen aus guten Gründen, übrigens auch unterstützt (C)  
>gen Sie denn zu der Feststellung des SPD-Fraktionschefs, von der eigenen Regierung, auf diesen globalen  
>Arbeitsdie in der „FAZ“ vom 13. April wiedergegeben wird? Er markt drängt, dann ist die klassische Diskussion, zum  
>Beisoll gesagt haben, die SPD-Fraktion lege auch keinen all- spiel um Braindrain wie in den 70er-Jahren, hier völlig  
>zu großen Wert auf formale Hochschulabschlüsse, sondern fehl am Platze.   
>neben der Orientierung an formalen  
>Hochschulabschlüssen allein, wie sie im Entwurf von Arbeitsminister Riester (Beifall bei Abgeordneten der SPD)  
>zunächst vorgesehen sei, werde daran gedacht, sich an den Indien profitiert davon, dass es eine starke Gruppe von  
>gezahlten Gehältern für Fachleute aus dem Nicht-EU- Indern gibt, die die Computerindustrie in Silicon Valley  
>Raum zu orientieren. Diese liegen ja wohl nicht sehr hoch. mit aufgebaut haben.  
>Wolf-Michael Catenhusen, Parl. Staatssekretär bei (Jörg Tauss [SPD]: Heute schon!)  
>der Bundesministerin für Bildung und Forschung: Der Wir werden auf die Dauer davon profitieren, dass in unse-  
>Teufel liegt immer im Detail, Frau Luft. ren Unternehmen hoch qualifizierte Spezialisten arbeiten,  
>Übrigens ist meine Prognose ganz klar: Es werden nicht die anschließend in ihren eigenen Ländern unsere  
>Gehauptsächlich Inder sein, sondern zu uns werden vor allem schäftspartner werden.   
>hoch qualifizierte Experten aus dem osteuropäischen (Beifall bei der SPD sowie des Abg. Matthias  
>Raum kommen. Das weiß doch jeder. Berninger [BÜNDNIS 90/DIE GRÜNEN])  
</details>

Das macht eine Auswertung natürlich zwangsweise fehlerbehaftet. Weil es aber schlicht unmöglich ist, alles probezulesen müssen wir über solche Fehler aber leider hinwegsehen und nehmen alle Protokolle gleich auf.

# Daten der Bundestagsabgeordneten 

Hier laden wir zum Nachschlagen von Sprechern ohne gegebene Parteizugehörigkeit (Staatssekretäre, Bundesminister etc.) eine Liste an Namen und Parteien, die im Notebook "get_abgeordneten_data" erstellt wurde.

In [81]:
import csv
bt_tuples = []
with open('./data/abgeordnete.csv', 'r', newline= '',encoding='utf8') as file:
    reader = csv.reader(file)
    next(reader) # skip first row
    for row in reader:
        bt_tuples.append(tuple(row))


Manche Sprecher werden trotzdem nicht gefunden, weil sie nie Teil des Bundestags waren. Das trifft auf manche Staatssekretäre(innen) und Minister(innen) zu.
Fälle, die besonders oft vorkamen, wurden händisch in get_abgeordneten_data hinzugefügt.

In [125]:
def find_party_by_name(name, tuple_list):
    tuple_list_sorted = sorted(tuple_list, key = lambda x: x[2], reverse = True) # we focus on 
    if name.startswith('Dr.'):
        name = name[4:]
    for item in tuple_list_sorted:
        if name in item[0]:
            if item[1] != 'CSU' and item[1] != 'CDU':
                return item[1]
            else:
                return 'CDU/CSU'  
    return "<unknown>"

print(find_party_by_name('Dr. Robert Habeck', bt_tuples))
print(find_party_by_name('Winfried Fockenberg', bt_tuples))
print(find_party_by_name('Olaf Scholz', bt_tuples))

GRÜNE
CDU/CSU
SPD


Das absteigende Sortieren nach Wahlperioden in der Methode hilft, Fehler zu vermeiden. Wir schauen uns (zumindest derzeit) hauptsächlich neue Daten an.  
Deshalb reicht es aus, die wenigen Namenskonflikte in der Liste zu behandeln, indem wir einfach den neueren Eintrag nehmen.

# Get information out of protocol

In [ ]:
speakers_not_found = []

In [116]:
def parse_protocol(path_to_json, path_to_output_dir):
    text = load_text(path_to_json)
    if load_text == 'no text found':
        print(f"Processed protocol {ntpath.basename(path_to_json)}. NO TEXT FOUND")
    
    count_zusammengefasst = 0 #debugging

    speeches_raw = re.split(speaker_match, text)[1:]

    speeches = []
    for i in range(0, len(speeches_raw), 5):
        if speeches_raw[i] != None and speeches_raw[i+1] != None:
            name = speeches_raw[i].strip()    
            party = speeches_raw[i+1].strip()
            if party == 'CSU' or party == 'CDU': # sometimes the the secretary tasked with writing everything down forgets to put CDU/CSU instead of CSU or CDU
                party = 'CDU/CSU'
        else:
            name = speeches_raw[i+2].replace('\n', ' ').strip() # sometimes the formatting is all over the place e.g. "Olaf\nScholz" instead of "Olaf Scholz" :)
            party = find_party_by_name(name, bt_tuples)

        speech = {
            'speaker':{
                'name':name,
                'party':party
            },
            'text': re.split(r'(\nVizepräs.{0,99}?:)|(\nPräsid.{0,99}?:)|(\nAnlage)',speeches_raw[i+4])[0].strip() # handle end of File (Anlagen) and interruptions by the Bundestagspräsident(in) or Vice Bundestagspräsident(in)
        }


        #debugging
        if party not in parties:
            print(f"    Error at {speech}")
            print(f"    Capture Groups: {speeches_raw[i:i+4]}")
            print(f"    Speaker not found. Either Speaker is not part of BT,  RegEx falsely matched expression or protocol has errors. Speech will not appear in parsed version.")
            speakers_not_found.append((ntpath.basename(path_to_json), name, party))
            continue
            # This is triggered mostly by guest-speakers and name-typos. Also, sometimes a state-secretary has not been member of the BT before becoming state-secretary

        #end_debugging
        
        if len (speeches) >=1 and speeches[-1]['speaker'] == speech['speaker']:
            speeches[-1]['text'] += '\n' + speech['text'] # append interrupted speeches by same speaker e.g. after interruptions by Bundestagspräsident(in)
            count_zusammengefasst +=1
        else:
            speeches.append(speech)

        
        comments = []
        # comments with known speaker
        for match in re.finditer(unterbrechung_match,speech['text']):
            comment = {
                'commentator': {
                    'name': match.group(1),
                    'party': match.group(2)
                },
                'text': match.group(3).strip(),
                'preceding_context': speech['text'][:match.start()] # include all text until comment for later training of LLM
            }
            comments.append(comment)
        
        # comments with unknown speaker
        for match in re.finditer(zuruf_match, re.sub('der LINKEN', 'DIE LINKE', speech['text'])):
            comment = {'commentator':{
                    'name': '<unknown>',
                    'party': match.group(1)
                },
                'text': match.group(2),
                'preceding_context': speech['text'][:match.start()]
            }
            comments.append(comment)

        speech['comments'] = comments

        # applause
        beifall = re.findall(beifall_match, speech['text'])
        beifall = ''.join(beifall)
        beifall = re.sub('der LINKEN', 'DIE LINKE', beifall)
        beifall_counts = {party: beifall.count(f' {party}') for party in parties}
        speech['applause'] = beifall_counts
  
    json_string = json.dumps(speeches, indent=4)
    print(f"Processed protocol {ntpath.basename(path_to_json)}. #speeches = {len(speeches)} #speeches_concatenated = {count_zusammengefasst}")
    # save to file
    with open(f"{path_to_output_dir}{ntpath.basename(path_to_json)}", "w") as json_file:
        json_file.write(json_string)


In [117]:
data_path = r'./data/opendata_api/protocols/'
output_path = r'./data/opendata_api/parsed_protocols/'

In [118]:
parse_protocol(data_path + '20_012_2022-01-14.json', output_path)

Processed protocol 20_012_2022-01-14.json. #speeches = 74 #speeches_concatenated = 27


In [127]:
all_files = os.listdir(data_path)
print(f"Gesamtanzahl der Protokolle: {len(all_files)}")
c = 0
for filename in all_files:
    parse_protocol(data_path + filename, output_path)
    c+=1
    if c % 20 == 0:
        print(f"{c/len(all_files)*100}% der Protokolle verarbeitet")

Gesamtanzahl der Protokolle: 2065
Processed protocol 12_013_1991-03-12.json. #speeches = 72 #speeches_concatenated = 14
Processed protocol 12_014_1991-03-13.json. #speeches = 90 #speeches_concatenated = 28
Processed protocol 12_015_1991-03-14.json. #speeches = 96 #speeches_concatenated = 31
Processed protocol 12_016_1991-03-15.json. #speeches = 26 #speeches_concatenated = 4
Processed protocol 12_017_1991-03-20.json. #speeches = 78 #speeches_concatenated = 15
    Error at {'speaker': {'name': 'Wolfang Gröbl', 'party': '<unknown>'}, 'text': 'Im Gegenteil, \ndie Berücksichtigung der ökologischen Probleme \nführt zu dieser jetzt gewählten Va riante, im übrigen \nauch im Hinblick auf die Verbindung Straßburg-Kehl, \ndie zeitlich als Konkurrenzstrecke zu der von Ihnen \nnachgefragten Strecke gesehen werden muß.'}
    Capture Groups: [None, None, 'Wolfang Gröbl', 'Parl. Staatssekretär']
    Speaker not found. Either Speaker is not part of BT,  RegEx falsely matched expression or protocol has 

# Prepare Data for statistical Analysis

Sätze der Reden zählen und alles in ein Dataframe laden, das per csv abgespeichert werden kann.  
So wird das Laden und die Handhabung der für die Datenanalyse relevanten Unterbrechungen schneller, weil nur eine Dateioperation nötig ist. 

In [ ]:
from tqdm import tqdm
from spacy.lang.de import German


relevant_protocols = os.listdir('./data/opendata_api/parsed_protocols/')
nlp = German()
nlp.add_pipe('sentencizer')
speech_id =0

comment_list = []
for protocol_filename in tqdm(relevant_protocols):
    with open('data/opendata_api/parsed_protocols/'+protocol_filename) as f:
        protocol = json.load(f)
        for speech in protocol:
            speech_id +=1
            speaker_name = speech['speaker']['name']
            speaker_party = speech['speaker']['party']
            doc = nlp(speech['text'])
            speech_len_sents = sum(1 for sent in doc.sents)
            if 'comments' in speech:
                for comment in speech['comments']:
                    comment_list.append([comment['text'],
                                  comment['commentator']['party'],
                                  comment['commentator']['name'],
                                  speaker_party,
                                  speaker_name,
                                  protocol_filename[7:-5],
                                  speech_len_sents, 
                                  speech_id
                                ])
columns=['comment_text', 'comment_party', 'comment_name',  'interrupted_speaker_party','interrupted_speaker', 'date', 'speech_len_sents', 'speech_id']
df_comments = pd.DataFrame(comment_list, columns = columns)


In [ ]:
df_comments.head(-20)

Leider sind noch Zeilenumbrüche enthalten...

In [ ]:
df_comments['comment_text'] = df_comments['comment_text'].str.replace('\n', ' ')
df_comments['comment_name'] = df_comments['comment_name'].str.replace('\n', ' ')
df_comments['interrupted_speaker'] = df_comments['interrupted_speaker'].str.replace('\n', ' ')

Leider schreibt der Protokollant/die Protokollantin nicht immer einheitlich mit...

In [ ]:
df_comments.loc[df_comments['comment_party'].isin(['CDU', 'CSU']), 'comment_party'] = 'CDU/CSU'

df_comments.loc[df_comments['interrupted_speaker_party'].isin(['CDU', 'CSU']), 'interrupted_speaker_party'] = 'CDU/CSU'

In [ ]:
df_comments.to_csv('./data/comments.csv') # Abspeichern

# Debugging

In [140]:

from tqdm import tqdm


relevant_protocols = [filepath for filepath in os.listdir('./data/opendata_api/parsed_protocols/') if filepath.endswith('.json')]
speech_id =0

comment_list = []
for protocol_filename in tqdm(relevant_protocols):
    with open('data/opendata_api/parsed_protocols/'+protocol_filename) as f:
        protocol = json.load(f)
        for speech in protocol:
            speech_id +=1
            speaker_name = speech['speaker']['name']
            speaker_party = speech['speaker']['party']
            if 'comments' in speech:
                for comment in speech['comments']:
                    if(len(comment['preceding_context'])< 300):
                        context_len = len(comment['preceding_context'])
                    else:
                        context_len=300
                    comment_list.append([comment['text'],
                                  comment['commentator']['party'],
                                  comment['commentator']['name'],
                                  speaker_party,
                                  speaker_name,
                                  protocol_filename[7:-5],
                                  speech_id, 
                                  comment['preceding_context'][-context_len:].replace('\n', ' ')
                                ])
columns=['comment_text', 'comment_party', 'comment_name',  'interrupted_speaker_party','interrupted_speaker', 'date', 'context']
df_comments = pd.DataFrame(comment_list, columns = columns)

100%|██████████| 2075/2075 [00:23<00:00, 89.98it/s] 


ValueError: 7 columns passed, passed data had 8 columns

In [137]:
df_comments['comment_text'] = df_comments['comment_text'].str.replace('\n', ' ')
df_comments['comment_name'] = df_comments['comment_name'].str.replace('\n', ' ')
df_comments['interrupted_speaker'] = df_comments['interrupted_speaker'].str.replace('\n', ' ')

In [138]:
df_comments.loc[df_comments['comment_party'].isin(['CDU', 'CSU']), 'comment_party'] = 'CDU/CSU'

df_comments.loc[df_comments['interrupted_speaker_party'].isin(['CDU', 'CSU']), 'interrupted_speaker_party'] = 'CDU/CSU'

In [139]:
df_comments.to_csv('./data/comments_for_labeling.csv') # Abspeichern

In [44]:
data_path = r'data\opendata_api\protocols\12_004_1991-01-18.json'
data_path = r'data\opendata_api\protocols\16_092_2007-03-30.json'
data_path = r'data\opendata_api\protocols\20_114_2023-07-05.json'
data_path = r'data\opendata_api\protocols\20_115_2023-07-06.json'
data_path = r'data\opendata_api\protocols\13_182_1997-06-13.json'
data_path = r'data\opendata_api\protocols\14_099_2000-04-13.json'
data_path = r'.\data\opendata_api\protocols\20_169_2024-05-16.json'
data_path= r'.\data\opendata_api\protocols\12_113_1992-10-15.json'
output_path = './data/opendata_api/parsed_protocols/'
parse_protocol(data_path, output_path)

Processed protocol 12_113_1992-10-15.json. #speeches = 119 #speeches_concatenated = 33


In [63]:
buendnis_match = re.compile(r'(BÜND(-\n?)?NIS((-\n?)?SES)?\s*?90\/\s*(DIE\s*?)?GRÜ(-\n?)?NEN?)', flags=re.UNICODE|re.IGNORECASE)

In [73]:
text = load_text(data_path)
print(f"loaded text from {data_path}")
print(re.findall(buendnis_match, text))
text = re.sub(buendnis_match, 'GRÜNE', text)
print(re.findall(buendnis_match, text))

with open('./data/test/' + ntpath.basename(data_path)[:-4]+ 'txt', 'w', encoding = 'utf8') as f:
        f.write(text)

loaded text from .\data\opendata_api\protocols\12_113_1992-10-15.json
[]
[]


In [106]:
datapath = r'data\opendata_api\protocols\14_099_2000-04-13.json'

with open(datapath, 'r', encoding = 'utf8') as f:
        text = json.load(f)['text']

with open('./data/examples/' + ntpath.basename(data_path)[:-4]+ 'txt', 'w', encoding = 'utf8') as f:
        f.write(text)

In [132]:
speakers_not_found = [snf[:2] for snf in speakers_not_found]
print(speakers_not_found[1])

('12_023_1991-04-25.json', 'Rainer Runke')


In [134]:
import csv

with open('./data/speakers_not_part_of_bt.csv','w', encoding='utf8', newline='') as out:
    csv_out=csv.writer(out)
    csv_out.writerows(speakers_not_found) 


In [136]:
parsed_path = './data/opendata_api/parsed_protocols/'
n_speeches = 0
for filename in os.listdir(parsed_path):
    with open(parsed_path+filename) as f:
        protocol = json.load(f)
    n_speeches += len(protocol)
    

In [137]:
n_speeches

223500

In [138]:
len(speakers_not_found)/n_speeches

0.001087248322147651

# Sonderfälle

## Fehler in Protokollen:

###data\test\14_099_2000-04-13.txt Zeile ab 2503

manche Redner wie z.B. amtierende Minister, Staatssekretäre etc. fallen aus dem Raster heraus und es steht keine Partei dahinter -> Wir wollen trotzdem die Parteien anmerken  
-> Abgleich mit XML-Stammdatenliste. Hier besonders angenehm: Schema: <p>"&lt;Vorname> &lt;Name>, &lt;Titel>:" </p> -> Das Ganze auch OHNE Doktortitel bei z.B. Dr. Robert Habeck -> Auslesen von Stammdaten xml

Sonderfall Einwürfe bei 1951-Protokollen. und 1971 "(Abg. <nachname>: <text>)
1951: Auch neuer Textanfang: (Dr.)? <nachname> (<partei>) (<Wahlkreis>)? :
1971 + 1981: (Dr.)? <nachname> (<partei>) (<Wahlkreis>)?: